In [1]:
from bm25 import BM25Retriever
from vdb import FaissRetriever
from rrf import rrf_fusion
from langchain_ollama import ChatOllama
from typing import Annotated, TypedDict, List, Union, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from operator import add
from langgraph.graph import StateGraph, END, START
from sentence_transformers import CrossEncoder
from langchain_core.prompts import ChatPromptTemplate

d:\Miniconda\envs\agent\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from ragas import evaluate
from ragas.metrics.collections import faithfulness #, answer_relevance, context_precision, context_recall
from datasets import Dataset

In [3]:
import torch
print(f'{torch.cuda.is_available()} {torch.cuda.device_count()} {torch.cuda.get_device_name(0)}')

True 1 NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
class ReActState(TypedDict):
    # 放的是重排后的结果
    documents: List[dict]
    # 判断要不要重新生成，用在末尾的审查节点上
    should_generate_again: bool
    # 对话消息记录
    messages: Annotated[List[BaseMessage], add]
    # 在审查节点更新
    loop_count: int = 0
    # 审查意见
    feedback: str = ""

In [5]:
IS_DEBUG = True
def debug_info(s):
    if IS_DEBUG:
        print(f'[DEBUG] {s}')

In [6]:
NUM_CHAPTERS = 10
chunk_path = f'first{NUM_CHAPTERS}_chunks.json'
bm25_retriever = BM25Retriever(chunks_file=chunk_path)
faiss_retriever = FaissRetriever(chunks_file=chunk_path)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device='cuda')
llm = ChatOllama(model='qwen3:8b', num_ctx=4096, temperature=0, format='json')

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\DIVINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 1.100 seconds.
Prefix dict has been built successfully.


已加载 FAISS 索引，包含 62 个向量。


In [7]:
def retrieve_and_rerank_node(state: ReActState) -> ReActState:
    debug_info('到达retrieve_and_rerank节点')
    query = state['messages'][-1].content
    debug_info(f'检索查询内容：{query}')
    faiss_results = faiss_retriever.retrieve(query, top_k=20)
    bm25_results = bm25_retriever.retrieve(query, top_k=20)
    fusion_results = rrf_fusion([faiss_results, bm25_results], k=60, top_k=20)
    def rerank_docs(query, docs):
        texts = [doc['chunk_text'] for doc in docs]
        queries = [query] * len(texts)
        scores = reranker.predict(list(zip(queries, texts)))
        for doc, score in zip(docs, scores):
            doc['rerank_score'] = score
        docs.sort(key=lambda x: x['rerank_score'], reverse=True)
        return docs
    fusion_results = rerank_docs(query, fusion_results)[:3]
    return {
        "documents": fusion_results
    }

def doc_transform(docs: List[dict]) -> str:
    return '\n\n'.join([
        f'--- chapter {doc["chpt_id"]} - chunk {doc["chunk_id"]} ---\n{doc["chunk_text"]}'
        for doc in docs
    ])
# generate的生成要求：用prompt指导llm阅读documents，然后根据query生成str回答
def generate_node(state: ReActState) -> ReActState:
    debug_info('到达generate节点')
    docs = state['documents']
    debug_info(f'用于生成的文档数量：{len(docs)}')
    context_text = doc_transform(docs)
    debug_info(f'用于生成的文档内容：{context_text}')
    feedback_text = f"""【上一次生成的答案】：
{state['messages'][-1].content}
【对上一次生成的答案的审查意见】：
{state['feedback']}"""
    system_prompt = f"""你是一个熟读玄幻网络小说的专家，能够根据提供的上下文信息回答用户的问题。请仔细阅读以下提供的上下文内容，然后根据用户的问题生成准确且简洁的回答。
【上下文内容】：
{context_text}
【用户问题】：
{state['messages'][-1].content}
{feedback_text}
【输出要求】：
请直接输出 JSON 字符串，不得包含任何解释或 ```json 代码块。
格式示例：{{"answer": "阅读后产生的回答"}}
【给出回答】：
"""
    input_messages = [
        SystemMessage(content=system_prompt)
    ]
    response = llm.invoke(input_messages).content
    debug_info(f'生成内容：{response}')
    return {
        'messages': [AIMessage(content=response)]
    }
# feedback的生成要求：用prompt指导llm审查generate的回答，然后输出一个JSON字符串 {'should_generate_again': bool， 'reason': str}
def feedback_node(state: ReActState) -> ReActState:
    debug_info('到达feedback节点')
    context_text = doc_transform(state['documents'])
    last_response = state['messages'][-1].content
    feedback_instruction = f"""你是一个严谨的文档审查员。
你的任务是核对【模型回答】是否忠实于【参考资料】。
【判断准则】：
1. 【严防幻觉】：如果回答中包含参考资料中明确没有、或与资料内容直接矛盾的信息，设 should_generate_again 为 true。
2. 【核心缺失】：仅当回答漏掉了解决用户问题的“关键核心信息”时（例如问“谁治的”却没提医师），才设 should_generate_again 为 true。不要因为漏掉背景环境描写（如香炉、地毯）而要求重发。
3. 【容忍冗余】：如果回答已经准确回答了问题，即便语言不够优美或包含少量无关但正确的废话，只要不违背事实，应设 should_generate_again 为 false。
4. 【一致性优先】：如果本次回答已经根据之前的“审查意见”进行了修正，且修正后的内容符合参考资料，应设 should_generate_again 为 false，避免陷入无限循环。
【参考资料】:
{context_text}
【模型回答】:
{last_response}
【输出要求】：
请直接输出 JSON 字符串，不得包含任何解释或 ```json 代码块。
格式示例：{{"should_generate_again": false, "reason": "回答准确覆盖了关键情节"}}
"""
    input_messages = [
        SystemMessage(content=feedback_instruction)
    ]
    feedback_response = llm.invoke(input_messages).content
    debug_info(f'审查内容：{feedback_response}')
    import json
    feedback_json = json.loads(feedback_response)
    loop_count = state.get('loop_count', 0) + 1
    try:
        return {
            'should_generate_again': feedback_json['should_generate_again'],
            'feedback': feedback_json['reason'],
            'loop_count': loop_count
        }
    except KeyError:
        debug_info(f'反馈内容格式错误，默认设为不重新生成：{feedback_response}')
        return {
            'should_generate_again': False,
            'feedback': '反馈内容格式错误，默认设为不重新生成',
            'loop_count': loop_count
        }
def router(state: ReActState):
    if state['loop_count'] >= 3:
        return END
    if state['should_generate_again']:
        return 'generate'
    else:
        return END

In [8]:
workflow = StateGraph(ReActState)

workflow.set_entry_point('retrieve_and_rerank')
workflow.add_node('retrieve_and_rerank', retrieve_and_rerank_node)
workflow.add_node('generate', generate_node)
workflow.add_node('feedback', feedback_node)

workflow.add_edge('retrieve_and_rerank', 'generate')
workflow.add_edge('generate', 'feedback')
workflow.add_conditional_edges('feedback', router, {
    'generate': 'generate',
    END: END
})

app = workflow.compile()

initial_state: ReActState = {
    'documents': [],
    'should_generate_again': True,
    'messages': [],
    'loop_count': 0,
    'feedback': "",
}
# query = '被雷劈后，谁治疗了杨奇的伤势？' # “医治”和“治疗”的区别导致了草神医没有被搜到
query = '气功功法分为哪些级别？'
initial_state['messages'].append(HumanMessage(content=query))
result = app.invoke(initial_state)
print('最终结果：')
for msg in result['messages']:
    print(f'- {msg.content}')

[DEBUG] 到达retrieve_and_rerank节点
[DEBUG] 检索查询内容：气功功法分为哪些级别？


d:\Miniconda\envs\agent\lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


[DEBUG] 到达generate节点
[DEBUG] 用于生成的文档数量：3
[DEBUG] 用于生成的文档内容：--- chapter 10 - chunk 4 ---
“嗯。”
就在这时，杨奇大哥感受到了输入体内的真气，浑身一震，长长叹了一口气：“三弟，你的真气对我好像有一种特别舒服的效果，和父亲的真气感觉不一样。”
“是吗？”杨奇心中一喜：“难道自己的神象镇狱劲，对于影毒有克制的能力？现在真气不强，克制效果不明显，看来我得进一步努力修炼！”
“奇儿，你的真气看来很奇怪，经受过了雷电的淬炼，已经发生了本质的变化，从今天开始，我就教你我们杨家的绝学，不败王拳。”杨战傲然道：“既然你立下来了三个月击败杨鸿烈的誓言，那就要成全你。”
“不败王拳！”
杨奇心中一震，他早就听说，杨家家族之中，有一门压轴的绝学，近乎于王级气功。
天地之间，各种气功，分为下乘，中乘，上乘，王级气功，皇级气功，圣级气功，天级气功，神级气功。
“不败王拳”，就是介乎于上乘和王级之间的气功修行之法。
一般家族，拥有上乘气功修行之法，就已经足够可以支撑一个世家数百年，乃至于千年不灭。
王级气功，就可以开创出一个王朝。
皇级气功，圣级气功，天级气功是在传说中。
神级气功，顾名思义，不属于人类，属于神灵修行之法，连神都要修行的气功。
杨奇几乎是可以肯定，自己的“神象镇狱劲”，最起码也是天级气功，甚至还有可能是神级气功，因为这门气功修炼到达大成，身体八亿四千万微小颗粒，粒粒都有一头远古巨象之力，联合起来，成为神象，那是神灵的力量，不属于人间。
神级的气功出现在大陆上，怀璧其罪，是一场灾难，所以他对父亲都不敢说。
这种功法，多一个人知道就多一分危险，可以伏尸百万，灭其宗族都不能够外传。杨奇知道，只有自己有足够保护家族的力量时，才能够告诉父亲。

--- chapter 1 - chunk 1 ---
他的气功，已经修行到达了四段“炼气”的境界。
在丰饶大陆上，高手都修炼气功，气功修炼，分为九段。
一段为“养气”。
二段“运气”，搬运体内元气，游走经络。
三段“聚气”，把元气聚集起来藏在气海之中。
四段“炼气”，聚集起来元气之后，进行提炼，百炼成钢。
五段“暴气”，气功外放，隔空伤人。
一般的人修行，气功修行一段到四段，“养，运，聚，炼”，都是在体内搬运气流，只能够强身健体，但是到达了五段“暴气”境界，就

In [14]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import json

ragas_llm = LangchainLLMWrapper(
    ChatOllama(model='qwen3:8b', temperature=0)
)
print(f"last_msg: {result['messages'][-1].content}")
last_msg = json.loads(result['messages'][-1].content).get('answer', 'failed')
print(last_msg)

ragas_data = {
    'question': [query],
    'answer': [last_msg],
    'contexts': [[doc['chunk_text'] for doc in result['documents']]],
    'ground_truth': ['气功分为下乘，中乘，上乘，王级气功，皇级气功，圣级气功，天级气功，神级气功。']
}

dataset = Dataset.from_dict(ragas_data)

from ragas.metrics.collections import Faithfulness

score = evaluate(
    dataset,
    metrics=[
        Faithfulness(),
        # answer_relevance,
    ],
    llm=ragas_llm
)
print(score)

last_msg: {
  "answer": "气功功法分为九段：一段为‘养气’，二段‘运气’，三段‘聚气’，四段‘炼气’，五段‘暴气’，六段‘兵气’，七段‘象气’，八段‘化气’，九段‘气宗’。"
}
气功功法分为九段：一段为‘养气’，二段‘运气’，三段‘聚气’，四段‘炼气’，五段‘暴气’，六段‘兵气’，七段‘象气’，八段‘化气’，九段‘气宗’。


C:\Users\Divinitist\AppData\Local\Temp\ipykernel_46204\2579778806.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(


TypeError: Faithfulness.__init__() missing 1 required positional argument: 'llm'